In [ ]:
import os
import pandas as pd
from langchain_openai import ChatOpenAI

need_translation = False
if need_translation:
    df = pd.read_csv('lausanne_map_pack.csv')
    key_abs_path = '/Users/viaccoz/Documents/GitHub/europeana/api_keys/openapikey.txt'
    with open(key_abs_path, 'r') as f:
        cont = f.readlines()
    params = dict((v.replace('\n', '').split('=')) for v in cont)
    # TODO: make it so the output does not have "Here is the translation" at the beginning or similar contextual phrases.
    translation_prompt = """
    You are a professional translator specialized in historical and cartographic terminology. Translate the following {lang1} text into {lang2}, ensuring that all technical terms related to maps and history are accurately conveyed. Maintain the original meaning and context while adapting the language for clarity and precision in English.
    """

    client = ChatOpenAI(
        openai_organization=params['organization_id'],
        openai_api_key=params['api_key'],
        model='gpt-4.1',
        temperature=0.7
    )

    def translate_text(text, source_lang, target_lang):
        prompt = translation_prompt.format(lang1=source_lang, lang2=target_lang)
        response = client.predict(f"{prompt}\n\nText to translate:\n{text}")
        return response.strip()


    df['description_en'] = df.apply(lambda row: translate_text(row['description'], 'French', 'English'), axis=1)
    df['description_it'] = df.apply(lambda row: translate_text(row['description'], 'French', 'Italian'), axis=1)
    df['description_de'] = df.apply(lambda row: translate_text(row['description'], 'French', 'German'), axis=1)
    df['nom_fr'] = df.apply(lambda row: translate_text(row['nom'], 'English', 'French'), axis=1)
    df['nom_it'] = df.apply(lambda row: translate_text(row['nom'], 'English', 'Italian'), axis=1)
    df['nom_de'] = df.apply(lambda row: translate_text(row['nom'], 'English', 'German'), axis=1)
    df['paradata_process_fr'] = df.apply(lambda row: translate_text(row['paradata_process'], 'English', 'French'), axis=1)
    df['paradata_process_it'] = df.apply(lambda row: translate_text(row['paradata_process'], 'English', 'Italian'), axis=1)
    df['paradata_process_de'] = df.apply(lambda row: translate_text(row['paradata_process'], 'English', 'German'), axis=1)
    df['paradata_provenance_en'] = df.apply(lambda row: translate_text(row['paradata_provenance'], 'French', 'English'), axis=1)
    df['paradata_provenance_it'] = df.apply(lambda row: translate_text(row['paradata_provenance'], 'French', 'Italian'), axis=1)
    df['paradata_provenance_de'] = df.apply(lambda row: translate_text(row['paradata_provenance'], 'French', 'German'), axis=1)

    df.to_csv('lausanne_map_pack_translated.csv', index=False)
else:
    df = pd.read_csv('lausanne_map_pack_translated.csv')


In [17]:

map_script_template = '''
import sys
import os
# to retrieve the utils function used by all notebooks
parent_dir = os.path.abspath('../../../')
if parent_dir not in sys.path: sys.path.insert(0, parent_dir)
from utils.data_modeling import *
from utils.rde import RDE

BASE_SLUG = '{slug}'
MAP_SLUG = f"{{BASE_SLUG}}-map"
BEGIN_TR = {begin_tr}
END_TR = {end_tr}
tuple_TR = (datetime_obj_from_int_time(BEGIN_TR), datetime_obj_from_int_time(END_TR, match_to_end=True))
MAP_UUID = str(uuid.uuid5(VMAP_UUID5_NS, MAP_SLUG))
bm_slug = f"{{MAP_SLUG}}-base"
basemap_layer_uuid = str(uuid.uuid5(VMAP_UUID5_NS, bm_slug))
zoom_lvl= [11,21]

lausanne_area_uuids = get_area_uuids_from_slugs('../../areas/data', ['city-lausanne-area', 'country-switzerland-area'])

bm_layer = produce_layer_obj(basemap_layer_uuid,
                                        bm_slug,
                                         {{ 
                                         "en": ["Historical Map of Lausanne from {start_time}"],
                                         "fr": ["Carte historique de Lausanne de {start_time}"],
                                         "it": ["Mappa storica di Losanna dal {start_time}"],
                                         "de": ["Historische Karte von Lausanne aus dem Jahr {start_time}"]
                                         }},
                                         {{
                                         "en": ["{paradata_process}"],
                                         "fr": ["{paradata_process_fr}"],
                                         "it": ["{paradata_process_it}"],
                                         "de": ["{paradata_process_de}"]
                                         }},
                                         tuple_TR,
                                         MAP_UUID,
                                         is_vector=False,
                                         layer_configs=[produce_layer_config(
                                          str(uuid.uuid5(VMAP_UUID5_NS, f'{{bm_slug}}-config-1')),
                                          zoom_lvl=zoom_lvl,
                                          access_url="https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts/rest/TimeMachine:{slug}/raster/EPSG:900913x2/EPSG:900913x2:{{z}}/{{y}}/{{x}}?format=image/png",
                                          format='wmts'
                                          )]
                                        )

layers = [bm_layer]
layer_ids = [l['uuid'] for l in layers]
map_obj = produce_map_obj(MAP_UUID, 
                             MAP_SLUG,
                             {{
                              "fr": ["{nom_fr}"],
                              "it": ["{nom_it}"],
                              "de": ["{nom_de}"],
                              "en": ["{nom}"]
                             }},
                             {{
                             "fr": ["{description}"],
                             "it": ["{description_it}"],
                             "de": ["{description_de}"],
                             "en": ["{description_en}"]
                             }},
                             {{
                             "en": ["{paradata_process}"],
                             "fr": ["{paradata_process_fr}"],
                             "it": ["{paradata_process_it}"],
                             "de": ["{paradata_process_de}"]
                             }},
                              "https://image-timemachine.epfl.ch/iiif/3/venice%2Flayer_thumbnails%2F{slug}.png/full/max/0/default.jpg",
                              "1.0",
                              tuple_TR,
                              layer_ids,
                              areas_id=lausanne_area_uuids
                             )

save_data_file_if_different('', 'map', [map_obj], '{slug}_map', RDE.MAP.value)
save_data_file_if_different('', 'layers', layers, '{slug}_layers', RDE.LAYER.value)

'''

for index, row in df.iterrows():
    being_tr = int(row['start_time'])*10000 + 101
    end_tr = int(row['end_time'])*10000 + 1231
    script_content = map_script_template.format(
        slug=row['slug'],
        start_time=row['start_time'],
        begin_tr=being_tr,
        end_tr=end_tr,
        nom=row['nom'],
        description=row['description'],
        nom_fr=row['nom_fr'],
        nom_it=row['nom_it'],
        nom_de=row['nom_de'],
        description_en=row['description_en'],
        description_it=row['description_it'],
        description_de=row['description_de'],
        paradata_process_fr=row['paradata_process_fr'],
        paradata_process_it=row['paradata_process_it'],
        paradata_process_de=row['paradata_process_de'],
        paradata_process=row['paradata_process']
    )
    folder_name = row['slug']
    if not os.path.exists(folder_name):
        os.makedirs(folder_name)
    script_filename = f"produce_{row['slug'].replace('-', '_')}_map.py"
    with open(os.path.join(folder_name, script_filename), 'w') as script_file:
        script_file.write(script_content)

# compact version (in one file for everything)

In [23]:
import sys
import os
# to retrieve the utils function used by all notebooks
parent_dir = os.path.abspath('../../')
if parent_dir not in sys.path: sys.path.insert(0, parent_dir)
from utils.data_modeling import *
from utils.rde import RDE

layers = []
maps = []
for index, r in df.iterrows():
    slug = r['slug']
    begin_tr = int(r['start_time'])*10000 + 101
    end_tr = int(r['end_time'])*10000 + 1231
    BASE_SLUG = slug
    MAP_SLUG = f"{BASE_SLUG}-map"
    BEGIN_TR = begin_tr
    END_TR = end_tr
    tuple_TR = (datetime_obj_from_int_time(BEGIN_TR), datetime_obj_from_int_time(END_TR, match_to_end=True))
    MAP_UUID = str(uuid.uuid5(VMAP_UUID5_NS, MAP_SLUG))
    bm_slug = f"{MAP_SLUG}-base"
    basemap_layer_uuid = str(uuid.uuid5(VMAP_UUID5_NS, bm_slug))
    zoom_lvl= [11,21]

    lausanne_area_uuids = get_area_uuids_from_slugs('../areas/data', ['city-lausanne-area', 'country-switzerland-area'])

    bm_layer = produce_layer_obj(basemap_layer_uuid,
                                            bm_slug,
                                            {
                                            "en": [f"Historical Map of Lausanne from {r['start_time']}"],
                                            "fr": [f"Carte historique de Lausanne de {r['start_time']}"],
                                            "it": [f"Mappa storica di Losanna dal {r['start_time']}"],
                                            "de": [f"Historische Karte von Lausanne aus dem Jahr {r['start_time']}"]
                                            },
                                            {
                                            "en": [r['paradata_process']],
                                            "fr": [r['paradata_process_fr']],
                                            "it": [r['paradata_process_fr']],
                                            "de": [r['paradata_process_de']]
                                            },
                                            tuple_TR,
                                            MAP_UUID,
                                            is_vector=False,
                                            layer_configs=[produce_layer_config(
                                            str(uuid.uuid5(VMAP_UUID5_NS, f'{bm_slug}-config-1')),
                                            zoom_lvl=zoom_lvl,
                                            access_url=f"https://geo-timemachine.epfl.ch/geoserver/gwc/service/wmts/rest/TimeMachine:{slug}/raster/EPSG:900913x2/EPSG:900913x2:{{z}}/{{y}}/{{x}}?format=image/png",
                                            format='wmts'
                                            )]
                                            )

    layers.append(bm_layer)
    layer_ids = [bm_layer['uuid']]
    map_obj = produce_map_obj(MAP_UUID, 
                                MAP_SLUG,
                                {
                                "fr": [r['nom_fr']],
                                "it": [r['nom_it']],
                                "de": [r['nom_de']],
                                "en": [r['nom']]
                                },
                                {
                                "fr": [r['description']],
                                "it": [r['description_it']],
                                "de": [r['description_de']],
                                "en": [r['description_en']]
                                },
                                {
                                "en": [r['paradata_process']],
                                "fr": [r['paradata_process_fr']],
                                "it": [r['paradata_process_it']],
                                "de": [r['paradata_process_de']]
                                }, 
                                "https://image-timemachine.epfl.ch/iiif/3/venice%2Flayer_thumbnails%2F{slug}.png/full/max/0/default.jpg",
                                "1.0",
                                tuple_TR,
                                layer_ids,
                                areas_id=lausanne_area_uuids
                                )
    maps.append(map_obj)

folder_save_fp = f'lausanne-{min(df["start_time"])}-to-{max(df["end_time"])}-maps'
if not os.path.exists(folder_save_fp):
    os.makedirs(folder_save_fp)
save_data_file_if_different(folder_save_fp, 'map', maps, '{slug}_map', RDE.MAP.value)
save_data_file_if_different(folder_save_fp, 'layers', layers, '{slug}_layers', RDE.LAYER.value)